# Log4j NDG Extraction (Notebook Wrapper)

This notebook runs and inspects the file-level Network Dependency Graph (NDG) extraction pipeline implemented in `scripts/extract_log4j_ndg.py`. The script remains the single source of truth, so notebook and command-line runs generate the same artifacts.

The NDG contains one node per mapped PROMISE dataset class and typed dependency edges between Java files.

## 0) Setup Paths and Imports

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

# Resolve repo root whether the notebook is run from repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_log4j_ndg.py'
NDG_OUTPUT_DIR = REPO_ROOT / 'outputs' / 'log4j' / 'ndg'

print('Repository root:', REPO_ROOT)
print('Extraction script:', SCRIPT_PATH)
print('NDG output directory:', NDG_OUTPUT_DIR)

Repository root: /Users/iman/Desktop/python_sdp_gnn
Extraction script: /Users/iman/Desktop/python_sdp_gnn/scripts/extract_log4j_ndg.py
NDG output directory: /Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ndg


## 1) Run NDG Extraction

The extractor reads `outputs/log4j/log4j_preprocessed_standard.csv`, keeps rows with mapped Java files, parses source ASTs, resolves project types, and exports one directed multi-relational project graph.

It records these dependency relations:
- `EXTENDS`
- `IMPLEMENTS`
- `FIELD_TYPE`
- `PARAMETER_TYPE`
- `RETURN_TYPE`
- `OBJECT_CREATION`
- `METHOD_CALL`

In [2]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)

NDG extraction finished. nodes=119 edges=500 fallback_parses=4 validation_issues=0
summary=/Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ndg/ndg_summary.json
report=/Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ndg/ndg_report.md



## 2) Inspect Extraction Summary

In [3]:
summary = json.loads((NDG_OUTPUT_DIR / 'ndg_summary.json').read_text())
summary

{'dataset_rows': 135,
 'num_nodes': 119,
 'unmapped_rows': 16,
 'num_edges': 500,
 'distinct_file_pairs': 338,
 'node_feature_dim': 20,
 'edge_type_vocab_size': 7,
 'defective_nodes': 34,
 'non_defective_nodes': 85,
 'strict_parses': 115,
 'fallback_parses': 4,
 'parse_failures': 0,
 'validation_issues': 0}

In [4]:
graph_index = pd.read_csv(NDG_OUTPUT_DIR / 'graph_index.csv')
graph = json.loads((NDG_OUTPUT_DIR / 'graphs' / 'log4j.json').read_text())
nodes = pd.DataFrame(graph['nodes'])
edges = pd.DataFrame(graph['edges'])

print('Generated project graphs:', len(graph_index))
print('NDG nodes:', len(nodes))
print('Typed edges:', len(edges))
print()
print('Binary label distribution:')
print(nodes['label'].value_counts().sort_index())
graph_index


Generated project graphs: 1
NDG nodes: 119
Typed edges: 500

Binary label distribution:
label
0    85
1    34
Name: count, dtype: int64


,name,num_nodes,num_edges,feature_dim,graph_json,x_npy,y_npy,edge_index_npy,edge_type_npy
0,log4j,119,500,20,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...


In [5]:
edge_distribution = edges['edge_type'].value_counts().rename_axis('edge_type').reset_index(name='count')
edge_distribution

,edge_type,count
0,METHOD_CALL,180
1,PARAMETER_TYPE,93
2,FIELD_TYPE,71
3,OBJECT_CREATION,71
4,EXTENDS,37
5,RETURN_TYPE,27
6,IMPLEMENTS,21


## 3) Understand the Node Feature Vector

Each NDG node is a mapped Java file. Its stored feature vector contains the 20 standardized code metrics from the preprocessed PROMISE dataset:

```text
x(file) = [wmc, dit, noc, cbo, rfc, lcom, ca, ce, npm, lcom3, loc, dam, moa, mfa, cam, ic, cbm, amc, max_cc, avg_cc]
```

`name`, `source_path`, and `match_strategy` are metadata, not numeric features. The original `bug` count is also excluded from `x` to prevent target leakage. `y.npy` stores the binary classification target: `1` when `bug > 0`, otherwise `0`.

In [6]:
feature_names = json.loads((NDG_OUTPUT_DIR / 'feature_names.json').read_text())
pd.DataFrame({'feature_index': range(len(feature_names)), 'metric': feature_names})

,feature_index,metric
0,0,wmc
1,1,dit
2,2,noc
3,3,cbo
4,4,rfc
5,5,lcom
6,6,ca
7,7,ce
8,8,npm
9,9,lcom3


## 4) Inspect Graph Tensors

In [7]:
tensor_dir = NDG_OUTPUT_DIR / 'tensors'
x = np.load(tensor_dir / 'log4j_x.npy')
y = np.load(tensor_dir / 'log4j_y.npy')
edge_index = np.load(tensor_dir / 'log4j_edge_index.npy')
edge_type = np.load(tensor_dir / 'log4j_edge_type.npy')

print('x shape:', x.shape)
print('y shape:', y.shape)
print('edge_index shape:', edge_index.shape)
print('edge_type shape:', edge_type.shape)


x shape: (119, 20)
y shape: (119,)
edge_index shape: (2, 500)
edge_type shape: (500,)


In [8]:
EXAMPLE_NODE_ID = 0
print('Class:', nodes.loc[EXAMPLE_NODE_ID, 'name'])
print('Binary label:', y[EXAMPLE_NODE_ID])
print('Original bug count:', nodes.loc[EXAMPLE_NODE_ID, 'bug'])
pd.DataFrame({'metric': feature_names, 'value': x[EXAMPLE_NODE_ID]})


Class: org.apache.log4j.helpers.ISO8601DateFormat
Binary label: 0
Original bug count: 0


,metric,value
0,wmc,-0.460599
1,dit,2.584649
2,noc,-0.275087
3,cbo,-0.430914
4,rfc,-0.538731
5,lcom,-0.213664
6,ca,-0.200030
7,ce,-0.785464
8,npm,-0.102588
9,lcom3,1.625695


## 5) Inspect Typed Edges

`edge_index[:, i]` and `edge_type[i]` describe the same directed dependency. The readable graph JSON includes source and target class names.


In [9]:
edge_type_vocab = json.loads((NDG_OUTPUT_DIR / 'edge_type_vocab.json').read_text())
id_to_edge_type = {edge_type_id: edge_type_name for edge_type_name, edge_type_id in edge_type_vocab.items()}

print('Edge type vocabulary:')
display(pd.DataFrame(sorted(edge_type_vocab.items(), key=lambda item: item[1]), columns=['edge_type', 'edge_type_id']))
edges.head(20)

Edge type vocabulary:


,edge_type,edge_type_id
0,EXTENDS,0
1,IMPLEMENTS,1
2,FIELD_TYPE,2
3,PARAMETER_TYPE,3
4,RETURN_TYPE,4
5,OBJECT_CREATION,5
6,METHOD_CALL,6


,source,target,source_name,target_name,edge_type,edge_type_id
0,47,4,org.apache.log4j.Appender,org.apache.log4j.Layout,PARAMETER_TYPE,3
1,47,85,org.apache.log4j.Appender,org.apache.log4j.spi.ErrorHandler,PARAMETER_TYPE,3
2,47,66,org.apache.log4j.Appender,org.apache.log4j.spi.Filter,PARAMETER_TYPE,3
3,47,30,org.apache.log4j.Appender,org.apache.log4j.spi.LoggingEvent,PARAMETER_TYPE,3
4,19,47,org.apache.log4j.AppenderSkeleton,org.apache.log4j.Appender,IMPLEMENTS,1
5,19,4,org.apache.log4j.AppenderSkeleton,org.apache.log4j.Layout,FIELD_TYPE,2
6,19,4,org.apache.log4j.AppenderSkeleton,org.apache.log4j.Layout,PARAMETER_TYPE,3
7,19,25,org.apache.log4j.AppenderSkeleton,org.apache.log4j.Priority,FIELD_TYPE,2
8,19,25,org.apache.log4j.AppenderSkeleton,org.apache.log4j.Priority,METHOD_CALL,6
9,19,25,org.apache.log4j.AppenderSkeleton,org.apache.log4j.Priority,PARAMETER_TYPE,3


In [10]:
example_edge = graph['edges'][0]
example_edge


{'source': 47,
 'target': 4,
 'source_name': 'org.apache.log4j.Appender',
 'target_name': 'org.apache.log4j.Layout',
 'edge_type': 'PARAMETER_TYPE',
 'edge_type_id': 3}

## 6) Use the NDG in a GNN

The NDG is designed for relation-aware message passing. Each graph node already corresponds to one Java file and one defect label.

A typical model flow is:
1. Load `x`, `edge_index`, and `edge_type`.
2. Apply an `RGCNConv` layer or another typed-edge GNN layer.
3. Produce one NDG embedding for each file node.
4. Join file embeddings with pooled AST and CFG embeddings using the fully-qualified class names in `graphs/log4j.json`.
5. Predict each file's binary defect label from the fused representation.


## 7) Inspect Parse Recovery and Validation

In [11]:
parse_fallbacks = json.loads((NDG_OUTPUT_DIR / 'parse_fallbacks.json').read_text())
parse_failures = json.loads((NDG_OUTPUT_DIR / 'parse_failures.json').read_text())
validation_issues = json.loads((NDG_OUTPUT_DIR / 'validation_issues.json').read_text())

print('Legacy-normalized parse recoveries:', len(parse_fallbacks))
print('Unrecovered parse failures:', len(parse_failures))
print('Validation issues:', len(validation_issues))
pd.DataFrame(parse_fallbacks)

Legacy-normalized parse recoveries: 4
Unrecovered parse failures: 0
Validation issues: 0


,name,source_path,strict_error
0,org.apache.log4j.test.Finalize,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,JavaSyntaxError('')
1,org.apache.log4j.NDC,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,JavaSyntaxError('')
2,org.apache.log4j.Category,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,JavaSyntaxError('')
3,org.apache.log4j.PropertyConfigurator,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,JavaSyntaxError('')


## 8) Inspect the Generated Report

In [12]:
report_text = (NDG_OUTPUT_DIR / 'ndg_report.md').read_text()

try:
    from IPython.display import Markdown, display
    display(Markdown(report_text))
except ImportError:
    print(report_text)

# NDG Extraction Report for Log4j 1.0

## 1. Objective
This report describes the file-level Network Dependency Graph (NDG) extraction step for the Log4j 1.0 PROMISE dataset. The NDG captures static dependencies between mapped Java files so the software defect prediction model can learn project structure alongside per-file metrics, AST structure, and CFG behavior.

## 2. Graph Granularity
- The NDG is one project-level directed heterogeneous graph.
- Each node is one PROMISE dataset class with a mapped Java source file.
- Each directed edge `A -> B` means file `A` statically depends on file `B`.
- Dependencies to standard-library classes, external libraries, and source files without mapped PROMISE nodes are excluded because they cannot be represented as nodes in this file-level dataset graph.

## 3. Input Data
- Dataset: `outputs/log4j/log4j_preprocessed_standard.csv`
- Source root: `projects/log4j/logging-log4j1-v_1_0/src/java`
- Dataset rows: 135
- Rows with mapped source files: 119
- Rows without mapped source files: 16

## 4. Extraction Pipeline
- Each mapped Java file is parsed with `javalang` into a source AST.
- Legacy Log4j identifiers named `enum` or `assert` are deterministically renamed only when strict parsing fails.
- Imports, package declarations, and the source-tree class index resolve referenced types to project files.
- Dependencies are deduplicated by `(source, target, edge_type)`.
- Dependencies are stored as compact typed edges.

```text
Java source -> javalang AST -> project type resolution -> typed file dependencies -> project-level NDG tensors
```

## 5. Dependency Edge Types
| Edge type | ID | Exported edges |
| --- | ---: | ---: |
| `EXTENDS` | 0 | 37 |
| `IMPLEMENTS` | 1 | 21 |
| `FIELD_TYPE` | 2 | 71 |
| `PARAMETER_TYPE` | 3 | 93 |
| `RETURN_TYPE` | 4 | 27 |
| `OBJECT_CREATION` | 5 | 71 |
| `METHOD_CALL` | 6 | 180 |

| Edge type | Rule for creating `A -> B` |
| --- | --- |
| `EXTENDS` | Class `A` extends class `B`. |
| `IMPLEMENTS` | Class `A` implements interface `B`. |
| `FIELD_TYPE` | Class `A` declares a field whose type is `B`. |
| `PARAMETER_TYPE` | A method in class `A` accepts a parameter whose type is `B`. |
| `RETURN_TYPE` | A method in class `A` returns type `B`. |
| `OBJECT_CREATION` | A method in class `A` creates an object of type `B`. |
| `METHOD_CALL` | A method in class `A` calls a method through an expression statically resolved to `B`. |

The graph is multi-relational: a file pair can have multiple edges when different dependency rules apply.

## 6. Node Feature Vector
Each node uses the 20 standardized file metrics from the preprocessed PROMISE dataset:

```text
x(file) = [wmc, dit, noc, cbo, rfc, lcom, ca, ce, npm, lcom3, loc, dam, moa, mfa, cam, ic, cbm, amc, max_cc, avg_cc]
```

The original `bug` count is excluded from `x` to prevent target leakage. For binary defect prediction, `y.npy` stores `1` when `bug > 0`, otherwise `0`. Metadata columns `name`, `source_path`, and `match_strategy` are also excluded from `x`.

| Feature index | Metric |
| ---: | --- |
| 0 | `wmc` |
| 1 | `dit` |
| 2 | `noc` |
| 3 | `cbo` |
| 4 | `rfc` |
| 5 | `lcom` |
| 6 | `ca` |
| 7 | `ce` |
| 8 | `npm` |
| 9 | `lcom3` |
| 10 | `loc` |
| 11 | `dam` |
| 12 | `moa` |
| 13 | `mfa` |
| 14 | `cam` |
| 15 | `ic` |
| 16 | `cbm` |
| 17 | `amc` |
| 18 | `max_cc` |
| 19 | `avg_cc` |

## 7. Tensor Files and Semantics
| File | Shape | Meaning |
| --- | --- | --- |
| `tensors/log4j_x.npy` | `[119, 20]` | Standardized file-metric node features. |
| `tensors/log4j_y.npy` | `[119]` | Binary defective/non-defective labels (`bug > 0`). |
| `tensors/log4j_edge_index.npy` | `[2, 500]` | Directed file dependency connectivity. |
| `tensors/log4j_edge_type.npy` | `[500]` | Typed relation id aligned with each edge column. |

`edge_index[:, i]` and `edge_type[i]` describe the same directed relation.

## 8. How to Use the NDG in a GNN
The NDG already has one node per Java file, matching the target prediction granularity. Use a relational GNN such as `RGCNConv` so each file representation is updated from typed dependency edges. Later, fuse each NDG file embedding with pooled AST and CFG representations joined by fully-qualified class name.

```python
import numpy as np

root = 'outputs/log4j/ndg/tensors'
x = np.load(f'{root}/log4j_x.npy')
y = np.load(f'{root}/log4j_y.npy')
edge_index = np.load(f'{root}/log4j_edge_index.npy')
edge_type = np.load(f'{root}/log4j_edge_type.npy')

# x.shape          == [num_files, 20]
# y.shape          == [num_files]
# edge_index.shape == [2, num_edges]
# edge_type.shape  == [num_edges]
```

## 9. Output Files
- `outputs/log4j/ndg/graph_index.csv`: index row for the generated project-level graph.
- `outputs/log4j/ndg/graphs/log4j.json`: readable NDG nodes and typed edges.
- `outputs/log4j/ndg/tensors/log4j_x.npy`: node metric feature matrix.
- `outputs/log4j/ndg/tensors/log4j_y.npy`: binary defect label vector.
- `outputs/log4j/ndg/tensors/log4j_edge_index.npy`: directed edge connectivity.
- `outputs/log4j/ndg/tensors/log4j_edge_type.npy`: typed edge ids.
- `outputs/log4j/ndg/feature_names.json`: stable node feature order.
- `outputs/log4j/ndg/edge_type_vocab.json`: stable edge type vocabulary.
- `outputs/log4j/ndg/parse_fallbacks.json`: strict parser failures recovered by legacy normalization.
- `outputs/log4j/ndg/parse_failures.json`: unrecovered source parse failures.
- `outputs/log4j/ndg/validation_issues.json`: graph validation results.
- `outputs/log4j/ndg/ndg_summary.json`: global extraction statistics.

## 10. Results
| Metric | Value |
| --- | ---: |
| Dataset rows | 135 |
| NDG nodes | 119 |
| Unmapped dataset rows excluded | 16 |
| Typed NDG edges | 500 |
| Distinct connected file pairs | 338 |
| Strict parses | 115 |
| Legacy-normalized fallback parses | 4 |
| Unrecovered parse failures | 0 |
| Node feature dimension | 20 |
| Defective nodes (`bug > 0`) | 34 |
| Validation issues | 0 |

## 11. Validation
The generated tensors are checked for shape consistency, finite node features, binary labels, valid edge bounds, supported relation ids, unique typed edges, and absence of file-level self edges.

- Validation issues found: 0

## 12. Notes and Limitations
- Dependencies are static and file-level. They do not prove runtime execution.
- Method-call resolution is intentionally conservative. Calls without a resolvable receiver type are omitted.
- Dependencies to non-PROMISE files are filtered because the final graph nodes must align with mapped rows.
- Legacy normalization changes only parser input identifiers; it does not alter source files.

### 12.1 Legacy-Normalized Parses
| Class | Source path |
| --- | --- |
| `org.apache.log4j.test.Finalize` | `/Users/iman/Desktop/python_sdp_gnn/projects/log4j/logging-log4j1-v_1_0/src/java/org/apache/log4j/test/Finalize.java` |
| `org.apache.log4j.NDC` | `/Users/iman/Desktop/python_sdp_gnn/projects/log4j/logging-log4j1-v_1_0/src/java/org/apache/log4j/NDC.java` |
| `org.apache.log4j.Category` | `/Users/iman/Desktop/python_sdp_gnn/projects/log4j/logging-log4j1-v_1_0/src/java/org/apache/log4j/Category.java` |
| `org.apache.log4j.PropertyConfigurator` | `/Users/iman/Desktop/python_sdp_gnn/projects/log4j/logging-log4j1-v_1_0/src/java/org/apache/log4j/PropertyConfigurator.java` |

### 12.2 Unrecovered Parse Failures
| Class | Failure |
| --- | --- |
| None | None |

## 13. Sample Visualization
The diagram below shows a compact high-connectivity subset of the generated file-level NDG.

```mermaid
graph LR
  N4["Layout#4"]
  N12["BasicConfigurator#12"]
  N19["AppenderSkeleton#19"]
  N25["Priority#25"]
  N28["LogLog#28"]
  N30["LoggingEvent#30"]
  N36["FileAppender#36"]
  N38["TextPaneAppender#38"]
  N44["Category#44"]
  N47["Appender#47"]
  N56["PropertyConfigurator#56"]
  N64["DOMConfigurator#64"]
  N79["PatternParser#79"]
  N91["OptionConverter#91"]
  N99["AsyncAppender#99"]
  N113["Hierarchy#113"]
  N47 -->|PARAMETER_TYPE| N4
  N47 -->|PARAMETER_TYPE| N30
  N19 -->|IMPLEMENTS| N47
  N19 -->|FIELD_TYPE| N4
  N19 -->|PARAMETER_TYPE| N4
  N19 -->|FIELD_TYPE| N25
  N19 -->|METHOD_CALL| N25
  N19 -->|PARAMETER_TYPE| N25
  N19 -->|METHOD_CALL| N28
  N19 -->|PARAMETER_TYPE| N30
  N99 -->|PARAMETER_TYPE| N47
  N99 -->|RETURN_TYPE| N47
  N99 -->|EXTENDS| N19
  N99 -->|METHOD_CALL| N28
  N99 -->|METHOD_CALL| N91
  N99 -->|METHOD_CALL| N30
  N99 -->|PARAMETER_TYPE| N30
  N12 -->|PARAMETER_TYPE| N47
  N12 -->|METHOD_CALL| N44
  N12 -->|OBJECT_CREATION| N36
  N12 -->|PARAMETER_TYPE| N25
  N12 -->|METHOD_CALL| N28
  N12 -->|METHOD_CALL| N91
  N44 -->|METHOD_CALL| N47
  N44 -->|PARAMETER_TYPE| N47
  N44 -->|RETURN_TYPE| N47
  N44 -->|FIELD_TYPE| N113
  N44 -->|METHOD_CALL| N113
  N44 -->|OBJECT_CREATION| N113
  N44 -->|PARAMETER_TYPE| N113
  N44 -->|RETURN_TYPE| N113
  N44 -->|FIELD_TYPE| N25
  N44 -->|METHOD_CALL| N25
  N44 -->|PARAMETER_TYPE| N25
  N44 -->|RETURN_TYPE| N25
  N44 -->|METHOD_CALL| N28
  N44 -->|OBJECT_CREATION| N30
  N44 -->|PARAMETER_TYPE| N30
  N36 -->|EXTENDS| N19
  N36 -->|PARAMETER_TYPE| N4
  N36 -->|METHOD_CALL| N28
  N36 -->|METHOD_CALL| N91
  N36 -->|METHOD_CALL| N30
  N36 -->|PARAMETER_TYPE| N30
  N113 -->|FIELD_TYPE| N44
```

## 14. Future Integration
The NDG uses the same fully-qualified class names as the AST and CFG indexes. This keeps syntax in AST nodes, control behavior in CFG edges, and file dependencies in NDG edges while allowing class-level multi-view fusion.
